# Consumer Loan Risk — Scorecard Prototype

A transparent credit-risk screening prototype using a public Lending Club-derived extract.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score


In [ ]:
df = pd.read_csv('../data/loans_full_schema.csv')
print(df.shape)
df.head()


In [ ]:
df['default_flag'] = df['loan_status'].isin(['Charged Off','Default']).astype(int)
df[['loan_status','default_flag']].head()


## DTI segmentation

In [ ]:
df['dti_band'] = pd.cut(df['debt_to_income'], [-.01,10,20,30,40,100], labels=['<10%','10-20%','20-30%','30-40%','>40%'])
dti_summary = df.groupby('dti_band', observed=False)['default_flag'].agg(['count','mean']).rename(columns={'count':'applications','mean':'default_rate'})
dti_summary['default_rate_pct'] = dti_summary['default_rate'] * 100
dti_summary


## Logistic-regression scorecard prototype

In [ ]:
features = ['debt_to_income','annual_income','loan_amount','interest_rate','grade','homeownership','verified_income','loan_purpose','term']
num = ['debt_to_income','annual_income','loan_amount','interest_rate']
cat = [c for c in features if c not in num]
X_train, X_test, y_train, y_test = train_test_split(df[features], df['default_flag'], test_size=.2, random_state=42, stratify=df['default_flag'])
preprocess = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)])
model = Pipeline([('preprocess', preprocess), ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))])
model.fit(X_train, y_train)
prob = model.predict_proba(X_test)[:,1]
print('ROC-AUC:', round(roc_auc_score(y_test, prob), 3))
print('Default recall at 0.50:', round(recall_score(y_test, (prob >= .5).astype(int)), 3))


## Policy caution

The extract has only a small number of charged-off records. The score is therefore a portfolio triage prototype and must not be used as an automatic approval or rejection rule without a larger, time-split historical sample and calibration review.